# Colab Simulation: Standard Continuous Gaussian BCF on DGP C: Tweedie Semicontinuous
- **Model**: Standard Continuous Gaussian BCF (`BCF-Linear`)
- **DGP**: DGP C: Tweedie Semicontinuous
- **Sample Size (N)**: 1000
- **Zero-Inflation Intercept (c_shift)**: 0.0
- **MCMC**: NBURN = 1000, NSIM = 1000
- **Simulations**: 100 seeds
- **Output**: CSV containing CATE and Hurdle metrics.


In [ ]:
# Install devtools and the zicbcf package from GitHub
install.packages("remotes", repos="https://cloud.r-project.org/")
if (!require("devtools")) {
  install.packages("devtools", repos="https://cloud.r-project.org/")
}
devtools::install_github("hugogobato/zicbcf")
library(zicbcf)

In [ ]:
N_SIM   <- 100L
N       <- 1000L
P       <- 5L
NBURN   <- 1000L
NSIM    <- 1000L
NTHIN   <- 1L

OUT_CSV <- "results_bcf_dgp_c_N1000.csv"
if (file.exists(OUT_CSV)) file.remove(OUT_CSV)

In [ ]:
generate_dgp_c <- function(n, p, seed, c_shift = 0.0) {
  set.seed(seed * 1000 + 42)
  X <- matrix(rnorm(n * p), n, p)
  colnames(X) <- paste0("X", 1:p)
  
  pi_x <- pnorm(-0.5 + 0.4 * X[, 1] + 0.3 * X[, 2]^2)
  Z    <- rbinom(n, 1, pi_x)
  
  # Log-mean parameter (halved treatment effect)
  log_mu0 <- 1.2 + c_shift + 0.8 * X[, 1] - 0.4 * X[, 3]
  log_mu1 <- 1.2 + c_shift + 0.8 * X[, 1] - 0.4 * X[, 3] + 0.3 + 0.15 * X[, 1]
  
  mu0_true  <- exp(log_mu0)
  mu1_true  <- exp(log_mu1)
  true_cate <- mu1_true - mu0_true
  
  mu_true  <- ifelse(Z == 1, mu1_true, mu0_true)
  phi_true <- 1.5
  
  lambda0_true <- 2 * sqrt(mu0_true) / phi_true
  lambda1_true <- 2 * sqrt(mu1_true) / phi_true
  
  lambda_true <- ifelse(Z == 1, lambda1_true, lambda0_true)
  N_latent    <- rpois(n, lambda_true)
  gamma_true  <- 0.5 * phi_true * sqrt(mu_true)
  
  Y <- rep(0, n)
  for (i in 1:n) {
    if (N_latent[i] > 0) {
      Y[i] <- rgamma(1, shape = N_latent[i], scale = gamma_true[i])
    }
  }
  
  p0_hurdle_true <- 1 - exp(-lambda0_true)
  p1_hurdle_true <- 1 - exp(-lambda1_true)
  true_hurdle_cate <- p1_hurdle_true - p0_hurdle_true
  
  list(y = Y, z = Z, x = X, pihat = pi_x, true_cate = true_cate, true_ate = mean(true_cate), 
       true_hurdle_cate = true_hurdle_cate, true_hurdle_ate = mean(true_hurdle_cate))
}
calc_cate_metrics <- function(cate_draws, true_c, ate_draws) {
  cate_est <- colMeans(cate_draws)
  cate_ci  <- apply(cate_draws, 2, quantile, probs = c(0.025, 0.975))
  
  rmse <- sqrt(mean((cate_est - true_c)^2))
  bias <- mean(cate_est - true_c)
  coverage <- mean(true_c >= cate_ci[1, ] & true_c <= cate_ci[2, ])
  correlation <- cor(cate_est, true_c)
  if (is.na(correlation)) correlation <- 0.0
  ci_length <- mean(cate_ci[2, ] - cate_ci[1, ])
  est_ate_mean <- mean(ate_draws)
  
  list(rmse=rmse, bias=bias, coverage=coverage, correlation=correlation, ci_length=ci_length, est_ate_mean=est_ate_mean)
}

In [ ]:
cat("=== Starting Standard BCF Simulation ===\n")
for (s in 1:N_SIM) {
  cat(sprintf("[Seed %d/%d] Generating and fitting...\n", s, N_SIM))
  d <- generate_dgp_c(N, P, seed = s, c_shift = 0.0)
  
  fit <- bcf_continuous_linear(
    y          = d$y,
    z          = d$z,
    x_control  = d$x,
    x_moderate = d$x,
    zhat       = d$pihat,
    nburn      = NBURN,
    nsim       = NSIM,
    nthin      = NTHIN,
    update_interval = 99999
  )
  
  cate_draws <- get_forest_fit(fit$moderate_fit, d$x)
  ate_draws  <- rowMeans(cate_draws)
  m <- calc_cate_metrics(cate_draws, d$true_cate, ate_draws)
  
  df_res <- data.frame(
    DGP = "DGP C: Tweedie Semicontinuous",
    Seed = s,
    True_ATE = d$true_ate,
    True_Hurdle_ATE = d$true_hurdle_ate,
    
    Linear_CATE_RMSE        = m$rmse,
    Linear_CATE_Abs_Bias    = abs(m$bias),
    Linear_CATE_Coverage    = m$coverage,
    Linear_CATE_Correlation = m$correlation,
    Linear_CATE_CI_Length   = m$ci_length,
    Linear_Est_ATE          = m$est_ate_mean,
    
    # Smear columns populated as NA for BCF-Linear notebooks
    Smear_CATE_RMSE         = NA,
    Smear_CATE_Abs_Bias     = NA,
    Smear_CATE_Coverage     = NA,
    Smear_CATE_Correlation  = NA,
    Smear_CATE_CI_Length    = NA,
    Smear_Est_ATE           = NA,
    
    Linear_Hurdle_RMSE        = NA,
    Linear_Hurdle_Abs_Bias    = NA,
    Linear_Hurdle_Coverage    = NA,
    Linear_Hurdle_Correlation = NA,
    Linear_Hurdle_CI_Length   = NA,
    Linear_Est_Hurdle_ATE     = NA,
    
    Smear_Hurdle_RMSE        = NA,
    Smear_Hurdle_Abs_Bias    = NA,
    Smear_Hurdle_Coverage    = NA,
    Smear_Hurdle_Correlation = NA,
    Smear_Hurdle_CI_Length   = NA,
    Smear_Est_Hurdle_ATE     = NA,
    stringsAsFactors = FALSE
  )
  
  write.table(df_res, OUT_CSV, sep=",", row.names=FALSE, col.names=!file.exists(OUT_CSV), append=TRUE)
  gc(verbose=FALSE)
}
cat("\n=== Finished Standard BCF Run ===\n")